# Deep Learning Methods - Code

This notebook will demonstrate how to use the transformers library from Hugging Face to fine-tune BART and T5 for our review summarization task. Our steps include:

- **Load and process our data:** Create a dataset class that includes tokenized inputs and targets
- **Define training logic:** Write training functions for BART and T5
- **Generate summaries:** Use our fine-tuned models to generate abstractive summaries and compare with gold summaries

In [2]:
# Install necessary libraries

!pip install -q rouge_score evaluate bert_score

In [3]:
# Import necessary libraries

import pandas as pd
import numpy as np
import os
import json
import random
import torch
from torch.utils.data import Dataset
from transformers import BartForConditionalGeneration, BartTokenizer, T5ForConditionalGeneration, T5Tokenizer
from transformers import EarlyStoppingCallback, TrainingArguments, Trainer, DataCollatorForSeq2Seq
import evaluate
import textwrap
from sklearn.model_selection import train_test_split

from rouge_score import rouge_scorer
from bert_score import score as bert_score

import gc

import warnings
warnings.filterwarnings('ignore')

In [4]:
# Set seeds for reproducibility

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [5]:
# Models are fine-tuned on GPU

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Load our processed data

train_df = pd.read_json('data/amasum_train_processed.json', orient='records', lines=True)
test_df = pd.read_json('data/amasum_test_processed.json', orient='records', lines=True)

In [7]:
# Create validation dataset (can be moved to EDA section)

train_df, valid_df = train_test_split(train_df, test_size=0.2, random_state=42)
train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

## Summarization

The following section will cover how we import pretrained BART and T5 models and fine-tune them with our data. Below we define which models we'll be using, load pretrained instances of them, and load their tokenizers:

In [8]:
bart_model_name = 'facebook/bart-large-cnn'
bart_model = BartForConditionalGeneration.from_pretrained(bart_model_name)
bart_tokenizer = BartTokenizer.from_pretrained(bart_model_name)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
t5_model_name = 't5-base'
t5_model = T5ForConditionalGeneration.from_pretrained(t5_model_name)
t5_tokenizer = T5Tokenizer.from_pretrained(t5_model_name)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


### Preparing our data

We will start by defining our inputs and targets for the models - this involves tokenizing our review subsets and gold summaries, then truncating and padding them to normalize their sizes.

In [10]:
# Tokenize and process our inputs and targets

def create_dataset(df, tokenizer, model):
    all_reviews = df['cluster_reviews'].apply(lambda x: " [SEP] ".join(x))

    if model == 'BART': # Short prefixes can help guide the model
      verdict_reviews = ["Summarize: " + r for r in all_reviews]
      pros_reviews = ["List the POSITIVE aspects: " + r for r in all_reviews]
      cons_reviews = ["List the NEGATIVE aspects: " + r for r in all_reviews]
    elif model == 'T5': # T5 was trained with task prefixes, so more explicit prompts help
      verdict_reviews = ["summarize the following customer reviews: " + r for r in all_reviews]
      pros_reviews = ["what are the positive aspects mentioned in these reviews? " + r for r in all_reviews]
      cons_reviews = ["what are the negative aspects mentioned in these reviews? " + r for r in all_reviews]


    verdicts = [v for v in df['verdict']]
    pros = [", ".join(p) for p in df['pros']] # Some pros are lists, so we join them into a string
    cons = [", ".join(c) for c in df['cons']] # Same for cons

    # Tokenize inputs and targets (padding will be handled later by the DataCollator)
    verdict_inputs = tokenizer(verdict_reviews, truncation=True, padding=False, max_length=1024)
    verdict_targets = tokenizer(verdicts, truncation=True, padding=False, max_length=80)

    pros_inputs = tokenizer(pros_reviews, truncation=True, padding=False, max_length=1024)
    pros_targets = tokenizer(pros, truncation=True, padding=False, max_length=80)

    cons_inputs = tokenizer(cons_reviews, truncation=True, padding=False, max_length=1024)
    cons_targets = tokenizer(cons, truncation=True, padding=False, max_length=80)

    return {
        'verdict_inputs': verdict_inputs,
        'verdict_targets': verdict_targets,
        'pros_inputs': pros_inputs,
        'pros_targets': pros_targets,
        'cons_inputs': cons_inputs,
        'cons_targets': cons_targets
    }

In [11]:
# Define a custom Dataset class

class SummarizationDataset(Dataset):
    def __init__(self, inputs, targets, tokenizer):
        self.input_ids = inputs.input_ids
        self.attention_mask = inputs.attention_mask
        self.labels = []

        for label_seq in targets.input_ids:
            # Replace pad_token_id with -100
            masked_labels = [
                token_id if token_id != tokenizer.pad_token_id else -100
                for token_id in label_seq
            ]
            self.labels.append(masked_labels)

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx]
        }

## Model Training

Before training, we define a "compute_metrics" function that evaluates model performance during training. The only difference between these two functions is the tokenizer, which we can't pass as an argument during training.

In [12]:
# Define a compute_metrics function

rouge_metric = evaluate.load("rouge")

def compute_metrics_bart(eval_pred):

    predictions, labels = eval_pred

    # Unpack logits if tuple
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # Convert logits to token IDs if needed
    if predictions.ndim == 3:
        predictions = np.argmax(predictions, axis=-1)

    # Convert labels: replace -100 with pad_token_id for decoding
    labels = np.where(labels != -100, labels, bart_tokenizer.pad_token_id)

    # Decode predictions and labels
    decoded_preds = bart_tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = bart_tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Strip whitespace
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    # Compute ROUGE scores
    result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels)

    # Optional: Return only F1 scores (most commonly used)
    return {
        "rouge1": result["rouge1"],
        "rouge2": result["rouge2"],
        "rougeL": result["rougeL"],
    }


def compute_metrics_t5(eval_pred):

    predictions, labels = eval_pred

    # Unpack logits if tuple
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # Convert logits to token IDs if needed
    if predictions.ndim == 3:
        predictions = np.argmax(predictions, axis=-1)

    # Convert labels: replace -100 with pad_token_id for decoding
    labels = np.where(labels != -100, labels, t5_tokenizer.pad_token_id)

    # Decode predictions and labels
    decoded_preds = t5_tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = t5_tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Strip whitespace
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    # Compute ROUGE scores
    result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels)

    # Optional: Return only F1 scores (most commonly used)
    return {
        "rouge1": result["rouge1"],
        "rouge2": result["rouge2"],
        "rougeL": result["rougeL"],
    }

Next, we'll define the function for fine-tuning our models. Tokenizers and training arguments will vary by model, but the overall training logic will be the same for both BART and T5. We will train three separate model instances - one for each target (verdict, pros, and cons) as these are distinct tasks.

In [13]:
# Training function

def train_model(train_inputs, valid_inputs, train_targets, valid_targets, target_type, training_args, compute_metrics_func, model):

    if model == 'BART':
      task_model = BartForConditionalGeneration.from_pretrained(bart_model_name)
      tokenizer = BartTokenizer.from_pretrained(bart_model_name)
    elif model == 'T5':
      task_model = T5ForConditionalGeneration.from_pretrained(t5_model_name)
      tokenizer = T5Tokenizer.from_pretrained(t5_model_name)


    train_dataset = SummarizationDataset(train_inputs, train_targets, tokenizer)
    eval_dataset = SummarizationDataset(valid_inputs, valid_targets, tokenizer)

    training_args = TrainingArguments(
        output_dir = f'./{target_type}_model',
        **training_args
    )

    print(f"Training {target_type} model...")

    trainer = Trainer(
        model=task_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics_func,
        data_collator=DataCollatorForSeq2Seq(tokenizer, model=task_model),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )
    trainer.train()

    return task_model

::: {dropdown} Code Explanation

```python
if model == 'BART':
  task_model = BartForConditionalGeneration.from_pretrained(bart_model_name)
  tokenizer = BartTokenizer.from_pretrained(bart_model_name)
elif model == 'T5':
  task_model = T5ForConditionalGeneration.from_pretrained(t5_model_name)
  tokenizer = T5Tokenizer.from_pretrained(t5_model_name)
```
This block initializes the model and tokenizer based on the input model name. It supports either BART or T5, both commonly used for text summarization tasks.

```python
train_dataset = SummarizationDataset(train_inputs, train_targets, tokenizer)
eval_dataset = SummarizationDataset(valid_inputs, valid_targets, tokenizer)
```
The training and validation datasets are prepared using a custom `SummarizationDataset` class, which tokenizes the input-target pairs for supervised training.

```python
training_args = TrainingArguments(
    output_dir = f'./{target_type}_model',
    **training_args
)
```
This sets up the training arguments using Hugging Face's `TrainingArguments` class, dynamically assigning the output directory and unpacking additional configurations provided via `training_args`.

```python
print(f"Training {target_type} model...")
```
A simple logging statement to indicate the start of model training for a specific target type (e.g., "verdict", "pros", or "cons").

```python
trainer = Trainer(
    model=task_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics_func,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=task_model),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)
trainer.train()
```
This initializes and runs Hugging Face's `Trainer`, handling training and evaluation. It uses early stopping to prevent overfitting and a data collator to batch inputs for sequence-to-sequence learning.

```python
return task_model
```
After training, the fine-tuned model is returned for downstream use, such as inference or further evaluation.

:::


Finally, we define a function to generate summaries from our fine-tuned models. It processing batches of input token tensors through separate models for each component, using task-specific generation settings.

In [14]:
# Create summaries from our fine-tuned models

def generate_summaries(
    verdict_inputs, pros_inputs, cons_inputs,
    verdict_model, pros_model, cons_model,
    tokenizer,
    max_length=64,
    batch_size=4,
):
    def process_in_batches(inputs, model, model_name, task_type):
        model.to(device)
        all_decoded = []

        print(f"Processing {model_name}...")

        for i in range(0, len(inputs["input_ids"]), batch_size):
            batch_inputs = {
                "input_ids": inputs["input_ids"][i:i+batch_size],
                "attention_mask": inputs["attention_mask"][i:i+batch_size]
            }

            examples = [
                {"input_ids": ids, "attention_mask": mask}
                for ids, mask in zip(batch_inputs["input_ids"], batch_inputs["attention_mask"])
            ]

            padded = tokenizer.pad(examples, padding=True, return_tensors="pt")
            padded = {k: v.to(device) for k, v in padded.items()}

            with torch.no_grad():
                # Task-specific generation parameters
                if task_type == "verdict":
                    outputs = model.generate(
                        **padded,
                        max_length=max_length,
                        min_length=20,
                        num_beams=3,
                        do_sample=False,
                        early_stopping=True,
                        pad_token_id=tokenizer.pad_token_id
                    )
                else:  # pros/cons
                    outputs = model.generate(
                        **padded,
                        max_length=max_length,
                        min_length=10,
                        num_beams=2,
                        do_sample=True,
                        temperature=0.7,
                        top_p=0.9,
                        early_stopping=True,
                        pad_token_id=tokenizer.pad_token_id
                    )

            batch_decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            all_decoded.extend(batch_decoded)

            del outputs, padded
            torch.cuda.empty_cache()

        model.to('cpu')
        torch.cuda.empty_cache()

        return all_decoded

    # Process each model type with task-specific parameters
    verdicts = process_in_batches(verdict_inputs, verdict_model, "verdicts", "verdict")
    pros = process_in_batches(pros_inputs, pros_model, "pros", "pros")
    cons = process_in_batches(cons_inputs, cons_model, "cons", "cons")

    summaries = [
        {"verdict": v.strip(), "pros": p.strip(), "cons": c.strip()}
        for v, p, c in zip(verdicts, pros, cons)
    ]

    return summaries

::: {dropdown} Code Explanation

```python
def process_in_batches(inputs, model, model_name, task_type):
    model.to(device)
    all_decoded = []

    print(f"Processing {model_name}...")
```
This inner function `process_in_batches` is responsible for generating summaries in batches. It begins by moving the model to the appropriate device (e.g., GPU) and initializes an empty list to hold decoded outputs.

```python
    for i in range(0, len(inputs["input_ids"]), batch_size):
        batch_inputs = {
            "input_ids": inputs["input_ids"][i:i+batch_size],
            "attention_mask": inputs["attention_mask"][i:i+batch_size]
        }

        examples = [
            {"input_ids": ids, "attention_mask": mask}
            for ids, mask in zip(batch_inputs["input_ids"], batch_inputs["attention_mask"])
        ]
```
The inputs are split into batches. Each batch consists of input IDs and attention masks, which are prepared into a list of dictionaries for padding and generation.

```python
        padded = tokenizer.pad(examples, padding=True, return_tensors="pt")
        padded = {k: v.to(device) for k, v in padded.items()}
```
This section uses the tokenizer to pad the examples to the same length and converts them into PyTorch tensors on the appropriate device.

```python
        with torch.no_grad():
            if task_type == "verdict":
                outputs = model.generate(
                    **padded,
                    max_length=max_length,
                    min_length=20,
                    num_beams=3,
                    do_sample=False,
                    early_stopping=True,
                    pad_token_id=tokenizer.pad_token_id
                )
            else:
                outputs = model.generate(
                    **padded,
                    max_length=max_length,
                    min_length=10,
                    num_beams=2,
                    do_sample=True,
                    temperature=0.7,
                    top_p=0.9,
                    early_stopping=True,
                    pad_token_id=tokenizer.pad_token_id
                )
```
This block runs the generation logic using different parameters depending on the task. Verdicts use deterministic decoding with beam search, while pros/cons use sampling-based decoding with temperature and top-p sampling for more diverse outputs.

```python
        batch_decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        all_decoded.extend(batch_decoded)

        del outputs, padded
        torch.cuda.empty_cache()
```
Generated sequences are decoded into strings and added to the result list. Intermediate tensors are deleted and the GPU cache is cleared to manage memory.

```python
    model.to('cpu')
    torch.cuda.empty_cache()
    return all_decoded
```
After processing all batches, the model is moved back to CPU and GPU memory is cleared again. The function returns the list of decoded summaries.

```python
verdicts = process_in_batches(verdict_inputs, verdict_model, "verdicts", "verdict")
pros = process_in_batches(pros_inputs, pros_model, "pros", "pros")
cons = process_in_batches(cons_inputs, cons_model, "cons", "cons")
```
Each input type (verdict, pros, cons) is processed using the corresponding model and decoding strategy via `process_in_batches`.

```python
summaries = [
    {"verdict": v.strip(), "pros": p.strip(), "cons": c.strip()}
    for v, p, c in zip(verdicts, pros, cons)
]

return summaries
```
The outputs from all three tasks are combined into structured dictionaries for each example, and returned as the final list of summaries.

:::


### BART

We will fine-tune BART first. With our model and tokenizer defined, we need to create the datasets for our model, define the training arguments, run the training function, then generate summaries.

In [15]:
bart_train_dataset = create_dataset(train_df, bart_tokenizer, 'BART')
bart_valid_dataset = create_dataset(valid_df, bart_tokenizer, 'BART')
bart_test_dataset = create_dataset(test_df, bart_tokenizer, 'BART')

In [16]:
bart_training_args = {
    "seed": 42, # Reproducibility
    "num_train_epochs": 4,  # Train for 4 full passes over the dataset
    "per_device_train_batch_size": 2,  # Small batch size to avoid OOM errors
    "per_device_eval_batch_size": 2,  # Match eval batch size to training for memory consistency
    "learning_rate": 1e-5,  # Low LR suitable for fine-tuning pretrained models
    "warmup_ratio": 0.1,  # Gradually increase LR over the first 10% of steps
    "weight_decay": 0.1,  # Apply regularization to reduce overfitting
    "save_steps": 500,  # Save model every 500 steps during training
    "logging_steps": 25,  # Log training metrics every 25 steps
    "eval_steps": 50,  # Evaluate model every 50 steps to monitor progress
    "save_total_limit": 2,  # Keep only the 2 most recent saved checkpoints
    "eval_strategy": "steps",  # Evaluate every N steps instead of each epoch
    "metric_for_best_model": "eval_loss",  # Use eval loss to determine the best checkpoint
    "greater_is_better": False,  # Lower eval loss is better
    "load_best_model_at_end": True,  # Reload best model after training ends
    "fp16": False,  # Disable mixed precision training for stability or hardware compatibility
    "gradient_accumulation_steps": 2,  # Accumulate gradients to simulate a larger batch size
    "dataloader_pin_memory": True,  # Speed up data transfer to GPU
    "report_to": "none"  # Disable integration with external logging services like WandB
}

In [17]:
bart_verdict_model = train_model(bart_train_dataset['verdict_inputs'],
                           bart_valid_dataset['verdict_inputs'],
                           bart_train_dataset['verdict_targets'],
                           bart_valid_dataset['verdict_targets'],
                           'verdict',
                           bart_training_args,
                           compute_metrics_bart,
                           'BART')

Training verdict model...


Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel
50,3.164700,2.868373,0.370424,0.100420,0.331253
100,3.002900,2.847992,0.371935,0.110070,0.337403
150,2.908500,2.806221,0.371303,0.109270,0.338418
200,2.907700,2.791591,0.374291,0.109425,0.339942
250,2.791100,2.756661,0.379284,0.113053,0.344573
300,2.467100,2.774155,0.377064,0.115045,0.344490
350,2.452100,2.775878,0.382021,0.120099,0.346934
400,2.547200,2.785773,0.378960,0.114877,0.345774


In [18]:
bart_pros_model = train_model(bart_train_dataset['pros_inputs'],
                        bart_valid_dataset['pros_inputs'],
                        bart_train_dataset['pros_targets'],
                        bart_valid_dataset['pros_targets'],
                        'pros',
                        bart_training_args,
                        compute_metrics_bart,
                        'BART')

Training pros model...


Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel
50,3.750200,3.465986,0.336453,0.073297,0.289036
100,3.450400,3.367995,0.339148,0.070068,0.291658
150,3.313400,3.269848,0.345505,0.077324,0.298767
200,3.312800,3.263814,0.340422,0.080361,0.297898
250,3.281700,3.243902,0.347923,0.079859,0.299517
300,2.933700,3.253317,0.344114,0.078616,0.300338
350,2.947000,3.246149,0.345595,0.082108,0.300704
400,3.032200,3.239731,0.346561,0.085476,0.302717
450,2.959300,3.234866,0.352475,0.081541,0.305268
500,2.928400,3.224736,0.346989,0.082534,0.302328


There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


In [19]:
bart_cons_model = train_model(bart_train_dataset['cons_inputs'],
                        bart_valid_dataset['cons_inputs'],
                        bart_train_dataset['cons_targets'],
                        bart_valid_dataset['cons_targets'],
                        'cons',
                        bart_training_args,
                        compute_metrics_bart,
                        'BART')

Training cons model...


Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel
50,3.699400,3.439490,0.312167,0.076434,0.284349
100,3.379600,3.371722,0.317190,0.076313,0.290478
150,3.097800,3.278874,0.317790,0.078302,0.293678
200,3.244800,3.225175,0.324564,0.082239,0.300087
250,3.117300,3.253376,0.328442,0.083491,0.303545
300,2.667800,3.253789,0.319705,0.077742,0.296632
350,2.632500,3.226929,0.320729,0.083891,0.295805


In [20]:
bart_generated = generate_summaries(
            bart_test_dataset['verdict_inputs'],
            bart_test_dataset['pros_inputs'],
            bart_test_dataset['cons_inputs'],
            bart_verdict_model,
            bart_pros_model,
            bart_cons_model,
            bart_tokenizer
)

Processing verdicts...
Processing pros...
Processing cons...


## T5

We follow the same process for T5:

In [21]:
t5_train_dataset = create_dataset(train_df, t5_tokenizer, 'T5')
t5_valid_dataset = create_dataset(valid_df, t5_tokenizer, 'T5')
t5_test_dataset = create_dataset(test_df, t5_tokenizer, 'T5')

In [22]:
t5_training_args = {
    "seed": 42, # Reproducibility
    "num_train_epochs": 4,  # Train for 4 full passes over the training data
    "per_device_train_batch_size": 2,  # Small batch size to prevent memory issues
    "per_device_eval_batch_size": 2,  # Match eval batch size for consistency and memory efficiency
    "learning_rate": 1e-5,  # Low learning rate for stable fine-tuning of a pretrained model
    "warmup_ratio": 0.1,  # Linearly increase learning rate during the first 10% of steps
    "weight_decay": 0.1,  # Apply L2 regularization to reduce overfitting
    "save_steps": 500,  # Save model checkpoint every 500 steps
    "logging_steps": 25,  # Log training metrics every 25 steps
    "eval_steps": 50,  # Evaluate the model every 50 steps
    "save_total_limit": 2,  # Keep only the 2 most recent checkpoints to save disk space
    "eval_strategy": "steps",  # Perform evaluation every N steps
    "metric_for_best_model": "eval_loss",  # Use evaluation loss to determine the best model
    "greater_is_better": False,  # Lower evaluation loss indicates better model performance
    "load_best_model_at_end": True,  # Load the best checkpoint after training finishes
    "fp16": False,  # Disable mixed precision (set to True if your GPU supports it for speed/memory gain)
    "gradient_accumulation_steps": 2,  # Accumulate gradients over 2 steps to simulate a larger batch size
    "dataloader_pin_memory": True,  # Speeds up data transfer to GPU
    "report_to": "none",  # Do not report to external logging platforms like WandB or TensorBoard
    "label_smoothing_factor": 0.1  # Apply label smoothing to improve generalization and prevent overconfidence
}

In [23]:
t5_verdict_model = train_model(t5_train_dataset['verdict_inputs'],
                         t5_valid_dataset['verdict_inputs'],
                         t5_train_dataset['verdict_targets'],
                         t5_valid_dataset['verdict_targets'],
                         'verdict',
                         t5_training_args,
                         compute_metrics_t5,
                         'T5')

Training verdict model...


Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel
50,5.254300,4.971951,0.333607,0.083381,0.298954
100,4.860000,4.532292,0.346865,0.090539,0.314333
150,4.548500,4.333602,0.358375,0.094557,0.327168
200,4.507400,4.240222,0.365358,0.097679,0.333923
250,4.343100,4.178550,0.368665,0.100878,0.338390
300,4.369400,4.147043,0.368028,0.098641,0.336690
350,4.317500,4.117182,0.371555,0.102312,0.341179
400,4.353100,4.101143,0.370626,0.103706,0.339809
450,4.320000,4.088073,0.370440,0.105256,0.339938
500,4.241500,4.076166,0.372807,0.105160,0.341223


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


In [24]:
t5_pros_model = train_model(t5_train_dataset['pros_inputs'],
                      t5_valid_dataset['pros_inputs'],
                      t5_train_dataset['pros_targets'],
                      t5_valid_dataset['pros_targets'],
                      'pros',
                      t5_training_args,
                      compute_metrics_t5,
                      'T5')

Training pros model...


Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel
50,5.896800,5.716592,0.318824,0.065022,0.274232
100,5.459200,5.082507,0.328709,0.067183,0.283756
150,4.954100,4.720569,0.330842,0.066736,0.287636
200,4.793400,4.590449,0.332020,0.067165,0.290091
250,4.715900,4.535870,0.331347,0.067496,0.291448
300,4.718900,4.508843,0.331946,0.068715,0.290687
350,4.642400,4.483404,0.335790,0.069996,0.294219
400,4.723500,4.467451,0.335787,0.070676,0.295040
450,4.634800,4.455984,0.335507,0.070179,0.295141
500,4.536300,4.446373,0.335362,0.070574,0.294812


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


In [25]:
t5_cons_model = train_model(t5_train_dataset['cons_inputs'],
                      t5_valid_dataset['cons_inputs'],
                      t5_train_dataset['cons_targets'],
                      t5_valid_dataset['cons_targets'],
                      'cons',
                      t5_training_args,
                      compute_metrics_t5,
                      'T5')

Training cons model...


Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel
50,6.539000,6.545050,0.271730,0.052653,0.252534
100,5.821500,5.517791,0.279016,0.055156,0.257701
150,5.165600,4.945169,0.287564,0.054020,0.264280
200,4.935500,4.732323,0.280795,0.054285,0.259093
250,4.720700,4.659407,0.283976,0.058650,0.264821
300,4.717900,4.622622,0.287735,0.061116,0.267189
350,4.641300,4.596971,0.286525,0.061919,0.267770
400,4.722100,4.580464,0.284881,0.060873,0.266918
450,4.677200,4.565955,0.284622,0.060790,0.266407
500,4.471300,4.551565,0.287054,0.061466,0.267629


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


In [26]:
t5_generated = generate_summaries(t5_test_dataset['verdict_inputs'],
                                  t5_test_dataset['pros_inputs'],
                                  t5_test_dataset['cons_inputs'],
                                  t5_verdict_model,
                                  t5_pros_model,
                                  t5_cons_model,
                                  t5_tokenizer
)

Processing verdicts...
Processing pros...
Processing cons...


## Evaluation

How do our generated summaries compare with the gold summaries?

In [27]:
gold_summaries = test_df[['verdict', 'pros', 'cons']].to_dict(orient='records')

In [28]:
# Define ROUGE evaluation functions

def evaluate_rouge_single(generated_summaries, gold_summaries):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []

    for gen, gold in zip(generated_summaries, gold_summaries):
        scores = scorer.score(gold, gen)
        rouge1_scores.append(scores['rouge1'].fmeasure)
        rouge2_scores.append(scores['rouge2'].fmeasure)
        rougeL_scores.append(scores['rougeL'].fmeasure)

    # Calculate averages
    avg_rouge1 = round(np.mean(rouge1_scores).item(), 4)
    avg_rouge2 = round(np.mean(rouge2_scores).item(), 4)
    avg_rougeL = round(np.mean(rougeL_scores).item(), 4)

    return {
        'rouge1': avg_rouge1,
        'rouge2': avg_rouge2,
        'rougeL': avg_rougeL
    }


def evaluate_rouge(generated, gold_data):
    # Extract generated parts
    gen_verdicts = [item['verdict'] for item in generated]
    gen_pros = [' '.join(item['pros']) for item in generated]
    gen_cons = [' '.join(item['cons']) for item in generated]

    # Extract gold parts
    gold_verdicts = [item['verdict'] for item in gold_data]
    gold_pros = [' '.join(item['pros']) for item in gold_data]
    gold_cons = [' '.join(item['cons']) for item in gold_data]

    # Evaluate each component
    verdict_rouge = evaluate_rouge_single(gen_verdicts, gold_verdicts)
    pros_rouge = evaluate_rouge_single(gen_pros, gold_pros)
    cons_rouge = evaluate_rouge_single(gen_cons, gold_cons)

    return {
        'verdict': verdict_rouge,
        'pros': pros_rouge,
        'cons': cons_rouge
    }

In [29]:
# Define BERTScore evaluation functions

def evaluate_bertscore_single(generated_summaries, gold_summaries, lang='en'):
    # Calculate BERTScore
    P, R, F1 = bert_score(generated_summaries, gold_summaries, lang=lang, verbose=True)

    return {
        'precision': round(P.numpy().mean().item(), 4),
        'recall': round(R.numpy().mean().item(), 4),
        'f1': round(F1.numpy().mean().item(), 4)
    }

def evaluate_bertscore(generated, gold_data, lang='en'):
    # Extract generated parts
    gen_verdicts = [item['verdict'] for item in generated]
    gen_pros = [' '.join(item['pros']) for item in generated]
    gen_cons = [' '.join(item['cons']) for item in generated]

    # Extract gold parts
    gold_verdicts = [item['verdict'] for item in gold_data]
    gold_pros = [' '.join(item['pros']) for item in gold_data]
    gold_cons = [' '.join(item['cons']) for item in gold_data]

    # Evaluate each component with BERTScore
    verdict_bert = evaluate_bertscore_single(gen_verdicts, gold_verdicts, lang)
    pros_bert = evaluate_bertscore_single(gen_pros, gold_pros, lang)
    cons_bert = evaluate_bertscore_single(gen_cons, gold_cons, lang)

    return {
        'verdict': verdict_bert,
        'pros': pros_bert,
        'cons': cons_bert
    }

### ROUGE Results

In [30]:
bart_rouge_results = evaluate_rouge(bart_generated, gold_summaries)

print("BART ROUGE Results:\n")
print(f"VERDICT:{bart_rouge_results['verdict']}")
print(f"PROS:{bart_rouge_results['pros']}")
print(f"CONS:{bart_rouge_results['cons']}")

BART ROUGE Results:

VERDICT:{'rouge1': 0.2235, 'rouge2': 0.0424, 'rougeL': 0.1654}
PROS:{'rouge1': 0.0122, 'rouge2': 0.0001, 'rougeL': 0.0122}
CONS:{'rouge1': 0.012, 'rouge2': 0.0001, 'rougeL': 0.012}


In [31]:
t5_rouge_results = evaluate_rouge(t5_generated, gold_summaries)

print("T5 ROUGE Results:\n")
print(f"VERDICT:{t5_rouge_results['verdict']}")
print(f"PROS:{t5_rouge_results['pros']}")
print(f"CONS:{t5_rouge_results['cons']}")

T5 ROUGE Results:

VERDICT:{'rouge1': 0.2166, 'rouge2': 0.0393, 'rougeL': 0.1688}
PROS:{'rouge1': 0.0161, 'rouge2': 0.0001, 'rougeL': 0.016}
CONS:{'rouge1': 0.0168, 'rouge2': 0.0, 'rougeL': 0.0168}


BART and T5 achieve comparable ROUGE results. ROUGE scores are very similar for generated verdicts, while T5's generated pros and cons seem to have slightly more n-gram overlap. These are slight improvements on our extractive models.

### BERTScore Results

In [32]:
bart_bertscore_results = evaluate_bertscore(bart_generated, gold_summaries)

print("BART BERTScore Results:\n")
print(f"VERDICT:{bart_bertscore_results['verdict']}")
print(f"PROS:{bart_bertscore_results['pros']}")
print(f"CONS:{bart_bertscore_results['cons']}")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

... (14 outputs left out)

  0%|          | 0/10 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/5 [00:00<?, ?it/s]

done in 4.12 seconds, 74.78 sentences/sec
BART BERTScore Results:

VERDICT:{'precision': 0.8764, 'recall': 0.8764, 'f1': 0.8763}
PROS:{'precision': 0.5994, 'recall': 0.7785, 'f1': 0.6772}
CONS:{'precision': 0.5992, 'recall': 0.803, 'f1': 0.6862}


In [33]:
t5_bertscore_results = evaluate_bertscore(t5_generated, gold_summaries)

print("T5 BERTScore Results:\n")
print(f"VERDICT:{t5_bertscore_results['verdict']}")
print(f"PROS:{t5_bertscore_results['pros']}")
print(f"CONS:{t5_bertscore_results['cons']}")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/10 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/5 [00:00<?, ?it/s]

done in 1.65 seconds, 186.27 sentences/sec


... (8 outputs left out)

  0%|          | 0/10 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/5 [00:00<?, ?it/s]

done in 3.10 seconds, 99.45 sentences/sec
T5 BERTScore Results:

VERDICT:{'precision': 0.8818, 'recall': 0.8725, 'f1': 0.877}
PROS:{'precision': 0.6097, 'recall': 0.7769, 'f1': 0.683}
CONS:{'precision': 0.6122, 'recall': 0.8024, 'f1': 0.6944}


BERTScores are also very comparable across all summaries, with T5 again showing slightly better performance for generated pros and cons.

In [ ]:
# Save results

results_to_save = {
    "gold_summaries": gold_summaries,
    "bart_rouge_results": bart_rouge_results,
    "t5_rouge_results": t5_rouge_results,
    "bart_bertscore_results": bart_bertscore_results,
    "t5_bertscore_results": t5_bertscore_results,
    "bart_generated": bart_generated,
    "t5_generated": t5_generated
}

# Save each dictionary as a JSON file
for name, data in results_to_save.items():
    file_path = f"results/{name}.json"
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

print("All results saved.")

All results saved.
